In [1]:
import lfox
import lfox.lattice as lat
import jax
import jax.numpy as jnp

import numpy as np

In [2]:
d = 2
L = 8

In [3]:
MyLat = lat.SquareLattice(dims=((L,)*d))
MyLat

In [4]:
phi_field = lat.LatticeField(MyLat)
phi_field.field = np.ones_like(phi_field.field)
phi_field.field[0,0] = 0.
phi_field.field = jnp.array(phi_field.field)
print(phi_field.field, phi_field)

[[0. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]] <lfox.lattice.LatticeField object at 0x120ae3210>


I0000 00:00:1695611804.850935       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


In [5]:
phi_field.nn_field(0)

Array([[1., 1., 1., 1., 1., 1., 1., 1.],
       [0., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1.]], dtype=float32)

In [6]:
phi_field + phi_field

AttributeError: 'LatticeField' object has no attribute 'copy'

In [6]:
def scalar_action(phi):
    S = 0.0
    F = phi.field
    for ax in range(d):
        S -= 2 * jnp.sum(F * phi.nn_field(ax))
    phi2 = F**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2 )

    return S

In [7]:
%time scalar_action(phi_field)

CPU times: user 98.2 ms, sys: 4.41 ms, total: 103 ms
Wall time: 102 ms


Array(-184., dtype=float32)

In [8]:
import lfox.evolution.hmc as lhmc

In [9]:
itest = lhmc.LeapfrogIntegrator(0.1, 10)
print(itest)

In [10]:
Stest = lhmc.Action({'phi': phi_field}, params={'lamb': 1.1})
print(Stest)

In [11]:
print(Stest.S())

0.0


In [12]:
class ScalarAction(lhmc.Action):
    def _S(self):
        return scalar_action(self.fields['phi'])

S2test = ScalarAction({'phi': phi_field}, params={'lamb': 1.1})



In [13]:
%time S2test.S()

CPU times: user 13.7 ms, sys: 1.26 ms, total: 15 ms
Wall time: 14.4 ms


Array(-184., dtype=float32)

In [14]:
HMCTest = lhmc.HMCEvolver(S2test, 0, itest)
print(HMCTest)

In [15]:
{fname: [] for fname in ['o', 'k']}

{'o': [], 'k': []}

In [16]:
HMCTest.evolve()

AttributeError: 'ScalarAction' object has no attribute 'copy'